# Tome 4, Chapter 7: The DFT and the FFT (Python companion)

Exercises first, solutions at the end. We write the FFT from scratch and time it, remove noise from a signal, compress an image the way JPEG does, multiply large integers exactly with a number-theoretic transform, and look at the flat spectrum of the Legendre symbol. Requirements: numpy and sympy.

*Before each solution, predict what the output will look like.*

## Exercises

**P7.1 (★) FFT from scratch.** Implement the recursive FFT of Theorem 7.9 in pure Python, check it against `np.fft.fft`, and compare its running time with the naive $O(n^2)$ DFT for $n=256,512,1024$. How should the ratio grow?

**P7.2 (★) CS application: denoising audio.** A $4096$-sample signal contains three tones (at $50$, $120$ and $300$ cycles per window) buried in Gaussian noise of standard deviation $0.8$. Transform, zero every coefficient of modulus below a threshold, transform back (Figure 7.2). Which threshold works best, and why do too low and too high thresholds both fail?

**P7.3 (★★) CS application: image compression.** Implement the DCT-II with one FFT (Proposition 7.12) and its inverse. Cut a $256\times256$ image into $8\times8$ blocks, apply the two-dimensional DCT to each block, keep only the $k$ largest coefficients per block, and reconstruct. Measure the quality by the peak signal-to-noise ratio $\mathrm{PSNR}=10\log_{10}\frac1{\mathrm{MSE}}$ for $k=1,3,6,10,20$.

**P7.4 (★★) CS application: exact big-integer multiplication.** Implement the iterative number-theoretic transform modulo $p=998\,244\,353$ (Theorem 7.11) with the bit-reversal permutation of Figure 7.1, and use it to multiply $3^{4000}$ by $7^{3000}$ digit by digit. Check the result against Python's own multiplication.

**P7.5 (★) A flat spectrum.** For several primes $p$, compute the Gauss sum $g$ and the DFT of the Legendre symbol, and check Proposition 7.13. What do you notice about $g$ when $p\equiv1$ and when $p\equiv3\pmod4$?

## Solutions

### P7.1

In [ ]:
import numpy as np, cmath, time

def dft_naive(a):
    n = len(a); w = np.exp(-2j * np.pi / n)
    return np.array([sum(a[k] * w**(j * k) for k in range(n)) for j in range(n)])

def fft(a):
    n = len(a)
    if n == 1:
        return list(a)
    E, O = fft(a[0::2]), fft(a[1::2])
    out = [0] * n
    for j in range(n // 2):
        t = cmath.exp(-2j * cmath.pi * j / n) * O[j]
        out[j], out[j + n // 2] = E[j] + t, E[j] - t
    return out

rng = np.random.default_rng(1)
a = rng.standard_normal(1024)
print("max |fft - numpy|      :", np.abs(np.array(fft(a)) - np.fft.fft(a)).max())
print("max |naive - numpy|    :", np.abs(dft_naive(a[:256]) - np.fft.fft(a[:256])).max())
for n in (256, 512, 1024):
    x = rng.standard_normal(n)
    t0 = time.perf_counter(); dft_naive(x); t1 = time.perf_counter(); fft(x); t2 = time.perf_counter()
    print(f"n = {n:4d}: naive {t1 - t0:7.3f} s, recursive FFT {t2 - t1:7.4f} s, ratio {(t1 - t0) / (t2 - t1):5.0f}")

Both agree with numpy to rounding error, the FFT more closely than the naive sum, because it performs fewer operations on each output. Doubling $n$ multiplies the naive time by $4$ and the FFT time by a bit more than $2$, so the ratio roughly doubles: it is $\frac{n^2}{\frac n2\log_2n}=\frac{2n}{\log_2n}$ up to constants. Numpy's FFT, written in C with an iterative in-place algorithm, is another factor of $100$ or so faster than this recursive Python version.

### P7.2

In [ ]:
n = 4096
t = np.arange(n) / n
clean = np.sin(2 * np.pi * 50 * t) + 0.5 * np.sin(2 * np.pi * 120 * t) + 0.3 * np.sin(2 * np.pi * 300 * t)
noisy = clean + 0.8 * rng.standard_normal(n)
A = np.fft.fft(noisy)
for thr in (100, 300, 600):
    den = np.fft.ifft(np.where(np.abs(A) > thr, A, 0)).real
    kept = int((np.abs(A) > thr).sum())
    print(f"threshold {thr:3d}: kept {kept:4d} coefficients, rms error {np.sqrt(np.mean((den - clean)**2)):.3f}"
          f"  (noise rms {np.sqrt(np.mean((noisy - clean)**2)):.3f})")
print("peaks found at frequencies:", sorted(int(j) for j in np.argsort(np.abs(A[:n // 2]))[-3:]))

The three tones have coefficients of modulus about $\frac n2$ times their amplitude ($2048$, $1024$, $614$), while white noise of standard deviation $\sigma$ spreads evenly with typical modulus $\sigma\sqrt n\approx51$ (Parseval: the energy $n\sigma^2$ is shared among $n$ coefficients). A threshold of $300$ keeps exactly the six coefficients $\pm50$, $\pm120$, $\pm300$, and the error falls from $0.80$ to $0.05$. At $100$, too many noise coefficients survive; at $600$, the weakest tone (coefficient about $614$, lowered by noise) is lost. Real denoisers use smarter thresholds and overlapping windows, but this is the principle.

### P7.3

In [ ]:
def dct2(x):                      # DCT-II via one FFT of the even extension (Proposition dct)
    N = len(x)
    y = np.concatenate([x, x[::-1]])
    Y = np.fft.fft(y)[:N]
    return (Y * np.exp(-1j * np.pi * np.arange(N) / (2 * N))).real / 2
def idct2(X):
    N = len(X); k = np.arange(N)
    C = np.cos(np.pi * np.outer(np.arange(N), 2 * k + 1) / (2 * N))
    return (X[0] / N) + (2 / N) * (X[1:, None] * C[1:]).sum(axis=0)
x = rng.standard_normal(8)
print("dct2 then idct2 recovers x:", np.allclose(idct2(dct2(x)), x))

# a synthetic 256 x 256 "photograph": smooth shading, a disc and some texture
yy, xx = np.mgrid[0:256, 0:256] / 256
img = 0.5 + 0.3 * np.sin(3 * xx + 2 * yy) + 0.25 * ((xx - .6)**2 + (yy - .4)**2 < .05) + 0.03 * np.sin(40 * xx) * np.cos(30 * yy)

def compress(img, keep):
    out = np.empty_like(img); total = 0
    for i in range(0, 256, 8):
        for j in range(0, 256, 8):
            B = img[i:i+8, j:j+8]
            C = np.array([dct2(r) for r in B]); C = np.array([dct2(c) for c in C.T]).T      # 2D DCT
            thr = np.sort(np.abs(C).ravel())[-keep]
            C = np.where(np.abs(C) >= thr, C, 0); total += int((C != 0).sum())
            R = np.array([idct2(c) for c in C.T]).T; out[i:i+8, j:j+8] = np.array([idct2(r) for r in R])
    return out, total
for keep in (1, 3, 6, 10, 20):
    rec, total = compress(img, keep)
    psnr = 10 * np.log10(1 / np.mean((rec - img)**2))
    print(f"keep {keep:2d} of 64 coefficients per block ({total / img.size:5.1%} of the data): PSNR {psnr:5.1f} dB")

Keeping $3$ of $64$ coefficients per block already gives about $38$ dB, a quality where differences are hard to see on smooth images. Why it works: by Proposition 7.12 the DCT of a block is the Fourier transform of its even extension, which has no artificial jump at the edges, so for smooth content the coefficients decay fast and the energy concentrates in the low frequencies. Blocks crossing the edge of the disc need more coefficients; with too few, the familiar JPEG blocking and ringing artifacts appear (the Gibbs phenomenon of Chapter 6, in two dimensions). JPEG then quantises the coefficients and compresses them losslessly with Huffman coding.

### P7.4

In [ ]:
import sys
sys.set_int_max_str_digits(0)            # Python limits int <-> str conversions by default
P, G = 998244353, 3                       # P = 119 * 2^23 + 1, 3 is a primitive root

def ntt(a, invert=False):
    a = list(a); n = len(a); j = 0
    for i in range(1, n):                 # bit-reversal permutation
        bit = n >> 1
        while j & bit:
            j ^= bit; bit >>= 1
        j |= bit
        if i < j: a[i], a[j] = a[j], a[i]
    length = 2
    while length <= n:                    # log2(n) passes of butterflies
        w = pow(G, (P - 1) // length, P)
        if invert: w = pow(w, P - 2, P)
        for s in range(0, n, length):
            wk = 1
            for k in range(length // 2):
                u, v = a[s + k], a[s + k + length // 2] * wk % P
                a[s + k], a[s + k + length // 2] = (u + v) % P, (u - v) % P
                wk = wk * w % P
        length <<= 1
    if invert:
        ninv = pow(n, P - 2, P); a = [x * ninv % P for x in a]
    return a

def multiply(x, y, base=10):
    da = [int(c) for c in str(x)[::-1]]; db = [int(c) for c in str(y)[::-1]]
    n = 1
    while n < len(da) + len(db): n <<= 1
    fa, fb = ntt(da + [0] * (n - len(da))), ntt(db + [0] * (n - len(db)))
    c = ntt([u * v % P for u, v in zip(fa, fb)], invert=True)
    carry, digits = 0, []
    for v in c:
        carry += v; digits.append(carry % base); carry //= base
    while carry: digits.append(carry % base); carry //= base
    return int("".join(map(str, digits[::-1])).lstrip("0") or "0")

x, y = 3**4000, 7**3000
print("1909-digit times 2536-digit product correct:", multiply(x, y) == x * y)
print("w = 3^((P-1)/2^23) has order 2^23:", pow(pow(3, (P - 1) >> 23, P), 1 << 22, P) == P - 1)

The product of a $1909$-digit and a $2536$-digit number is computed exactly with transforms of length $8192$. The coefficients of the digit convolution are at most $1909\cdot81<p$, so they are recovered exactly (Theorem 7.11), and the carries are propagated afterwards. Python's own integers use Karatsuba (Exercise 7.3); libraries like GMP switch to FFT methods for numbers of tens of thousands of digits, and the record computations of $\pi$ (Chapter 12) multiply numbers with trillions of digits this way.

### P7.5

In [ ]:
from sympy import legendre_symbol
for p in (5, 7, 11, 13, 101, 103):
    chi = np.array([0] + [legendre_symbol(k, p) for k in range(1, p)], dtype=float)
    g = (chi * np.exp(2j * np.pi * np.arange(p) / p)).sum()
    spec = np.abs(np.fft.fft(chi))
    print(f"p = {p:3d}: g = {g.real:+.6f} {g.imag:+.6f}i, |g|^2 = {abs(g)**2:8.4f}, "
          f"|chi_hat_j| for j != 0 all equal sqrt(p): {np.allclose(spec[1:], np.sqrt(p))}")

Every non-zero frequency of the Legendre symbol has modulus exactly $\sqrt p$: a flat spectrum, as Proposition 7.13 predicts. Moreover $g$ is real and equal to $+\sqrt p$ when $p\equiv1\pmod4$, and purely imaginary, equal to $+i\sqrt p$, when $p\equiv3\pmod4$. That $g^2=\pm p$ is algebra (Tome 1); that the sign is always $+$ took Gauss four years, and Chapter 8 proves it with Poisson summation.